In [ ]:

from collections import defaultdict
import itertools
import nltk
from nltk import ngrams
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize
import numpy as np
import pandas as pd
import pickle
import torch
import time
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Union, Optional
from torch.utils.data import DataLoader, Dataset

from dap_job_quality import config, PROJECT_DIR, logging, BUCKET_NAME
from dap_job_quality.getters.afs_data import get_eyp_ads, get_sim_occ_ads
from dap_job_quality.getters.models import sentence_classifier_pca, sentence_classifier_lr
from dap_job_quality.getters.data_getters import save_to_s3

# Load BERT model and tokenizer
model_name = config["sentence_model"]

nltk.download('stopwords')
nltk.download('punkt')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

model = sentence_classifier_lr()
pca = sentence_classifier_pca()

MAX_LENGTH = 81 # 99th percentile of token length of sentences
SAMPLE_SIZE = 'all'
SIMILARITY_THRESHOLD = 0

In [ ]:
class SentenceDataset(Dataset):
    def __init__(self, sentences, tokenizer, max_length=MAX_LENGTH):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return self.tokenizer(
            self.sentences[idx], padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt"
        )

# Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

# Function to embed sentences
def embed_sentences(sentences: List[str], model_name: str, batch_size: int = 32, device: str = 'cuda' if torch.cuda.is_available() else 'cpu') -> torch.Tensor:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    
    dataset = SentenceDataset(sentences, tokenizer)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    all_embeddings = []

    start_time = time.time()
    print(f'Process started at {start_time}')

    with torch.no_grad():
        for batch in dataloader:
            encoded_input = {key: val.squeeze().to(device) for key, val in batch.items()}
            model_output = model(**encoded_input)
            batch_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
            all_embeddings.append(batch_embeddings.cpu())

    elapsed_time = time.time() - start_time
    print(f"Batch size: {batch_size}, Time taken: {elapsed_time:.2f} seconds")

    return torch.cat(all_embeddings, dim=0)


def split_ngrams(text, length=6, n=4):
    if len(text.split()) > length:
        ngram_list = list(ngrams(text.split(), n))
    else:
        ngram_list = [text]
    return ngram_list

In [ ]:
eyp = get_eyp_ads()
sim_occs = get_sim_occ_ads()
all_job_ads = pd.concat([eyp, sim_occs], axis=0).drop_duplicates()

lookup = pd.read_csv('s3://open-jobs-lake/job_quality/keywords/keyword_lookup - v5.csv')

# Filter sentences which relate to job quality

In [ ]:
logging.info(len(all_job_ads))
all_job_ads = all_job_ads[all_job_ads['created']>='2023-01-01']
logging.info(len(all_job_ads))

In [ ]:
job_ads_sample['sentences'] = job_ads_sample['clean_description'].apply(sent_tokenize)
job_ads_sample = job_ads_sample.explode('sentences')
logging.info(f'{len(job_ads_sample)} sentences')

In [ ]:
len(job_ads_sample[job_ads_sample['clean_description']=='nan']) / len(job_ads_sample)

In [ ]:
# ad_embeddings = embed_sentences(job_ads_sample['sentences'].tolist(), model_name, 64)

In [ ]:
# save_to_s3(BUCKET_NAME, ad_embeddings.numpy(), 'job_quality/early_years/embeddings_2023.npy')

In [ ]:
import boto3
import numpy as np
import torch

def download_from_s3(bucket_name, object_name, file_name):
    s3_client = boto3.client('s3')
    s3_client.download_file(bucket_name, object_name, file_name)

def load_embeddings_numpy(file_name: str) -> torch.Tensor:
    np_array = np.load(file_name)
    tensor = torch.from_numpy(np_array)
    return tensor

OBJECT_NAME = 'job_quality/early_years/embeddings_2023.npy'
FILE_NAME = 'embeddings_2023.npy'

# Download the file from S3
download_from_s3(BUCKET_NAME, OBJECT_NAME, FILE_NAME)

# Load the embeddings from the .npy file and convert to tensor
ad_embeddings = load_embeddings_numpy(FILE_NAME)

In [ ]:
X_new_pca = pca.transform(ad_embeddings)

predictions = model.predict_proba(X_new_pca)[:, 1]

In [ ]:
job_ads_sample['job_quality'] = predictions

In [ ]:
job_ads_sample.head()

In [ ]:
job_ads_sample['job_quality'].hist(bins=100)

In [ ]:
job_ads_sample['job_quality'].describe()

In [ ]:
jq_sentences_df = job_ads_sample[job_ads_sample['job_quality']>=0.3]

In [ ]:
len(jq_sentences_df)

In [ ]:
len(jq_sentences_df) / len(job_ads_sample)

In [ ]:
save_to_s3(BUCKET_NAME, jq_sentences_df, 'job_quality/early_years/jq_sentences/jq_sentences_df_2023.parquet')

In [ ]:
from dap_job_quality.getters.data_getters import load_s3_data

test = load_s3_data(BUCKET_NAME, 'job_quality/early_years/jq_sentences/jq_sentences_df_2023.parquet')
test.head()

# Get ngrams from JQ sentences

In [ ]:
jq_sentences_df['ngrams'] = jq_sentences_df['sentences'].apply(lambda x: split_ngrams(x, 6, 4))
jq_sentences_df_long = jq_sentences_df.explode('ngrams')
jq_sentences_df_long['ngrams'] = jq_sentences_df_long['ngrams'].apply(lambda x: ' '.join(x) if isinstance(x, tuple) else x)
jq_sentences_df_long.head(20)

# Calculate cosine similarity of unique ngrams to target phrases

In [ ]:
ngram_counts = pd.DataFrame(jq_sentences_df_long['ngrams'].value_counts()).reset_index()
ngram_counts

In [ ]:
unique_ngrams = ngram_counts['ngrams']

In [ ]:
target_phrases = lookup['target_phrase'].tolist()

In [ ]:
# Embed the target phrases
target_embeddings = embed_sentences(target_phrases, model_name)
    
ngram_embeddings = embed_sentences(unique_ngrams, model_name)
        
similarities = cosine_similarity(ngram_embeddings, target_embeddings)

In [ ]:
# Find the index of the highest cosine similarity for each n-gram
max_indices = np.argmax(similarities, axis=1)

# Retrieve the corresponding target phrases
most_similar_phrases = [target_phrases[index] for index in max_indices]

most_similar_similarities = [similarities[i, index] for i, index in enumerate(max_indices)]

most_similar_pairs = list(zip(unique_ngrams, most_similar_phrases, most_similar_similarities))

matches = pd.DataFrame(most_similar_pairs, columns=['ngrams', 'target_phrase', 'cosine_similarity'])

In [ ]:
matches = pd.merge(matches, ngram_counts, on='ngrams', how='left')

In [ ]:
matches['count'].hist(bins=100)

In [ ]:
matches['count'].describe()

In [ ]:
jq_sentences_df_long = pd.merge(jq_sentences_df_long, matches, how='left', left_on='ngrams', right_on='ngrams')
jq_sentences_df_long.head()

In [ ]:
save_to_s3(BUCKET_NAME, jq_sentences_df_long, f'job_quality/early_years/jq_sentences/jq_sentences_2023_matched.parquet')
save_to_s3(BUCKET_NAME, matches, f'job_quality/early_years/jq_sentences/unique_ngrams_2023_matched.parquet')